# Сервис геопланирования — v3

Модуль распределяет визиты к точкам по рабочим дням месяца, группирует визиты
дня в компактные кластеры (≤ 12 точек) и строит порядок обхода.

**Ноутбук не содержит собственной логики**: все функции импортируются из
`project/src`, поэтому ноутбук, `run.py` и тесты используют один и тот же код.

Структура:
1. Загрузка и валидация данных
2. Анализ ёмкости — почему v2 теряла визиты
3. Кластеризация (и что было не так с DBSCAN)
4. План месяца и проверки по критериям приёмки ТЗ
5. Расстояния по дорогам и маршруты
6. Карта
7. Сравнение с v2 и выводы

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from src import (PlannerConfig, build_route, calculate_distance,
                 cluster_points, load_and_validate_data, run_pipeline,
                 summarize_points, visualize_map)
from src.config import DATA_PATH, OSRM_CACHE_PATH, OUTPUT_DIR
from src.distances import HaversineDistance
from src.road_geometry import OsrmRouter
from src.routing import path_length

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
pd.set_option("display.max_columns", 20)

# osrm — расстояния по дорогам (OSRM, ответы кешируются в cache/);
# haversine — по прямой, полностью офлайн
cfg = PlannerConfig(distance_backend="osrm")
cfg

PlannerConfig(working_days=22, max_points_per_cluster=12, min_points_per_cluster=8, max_clusters_per_manager_per_day=2, allowed_visits=(1, 2, 3, 4), cluster_fill_ratio=0.9, random_seed=42, distance_backend='osrm', road_detour_factor=1.3, avg_speed_kmh=50.0, service_time_min=15.0, max_work_hours=9.0, depot=(56.3269, 44.0052), default_manager=0, use_two_opt=True, map_road_geometry=True)

## 1. Загрузка и валидация

`load_and_validate_data` возвращает две таблицы: корректные точки и
отклонённые строки **с причиной**. Значения не исправляются молча (в v2
`clip(upper=2)` обрезал частоту, а `1.5` приводил к `TypeError`).

In [2]:
points, rejected = load_and_validate_data(DATA_PATH, cfg.allowed_visits,
                                          cfg.default_manager)
print(f"Корректных точек: {len(points)}, отклонено: {len(rejected)}")
points.head()

Переименованы колонки: {'lat': 'latitude', 'lon': 'longitude', 'n_visits': 'visits_per_month'}


Загружено строк: 633, корректных: 633, отклонено: 0


Корректных точек: 633, отклонено: 0


,point_id,latitude,longitude,visits_per_month,manager
0,ID1,55.371884,43.846878,1,0
1,ID2,55.358381,43.845061,1,0
2,ID3,55.374786,43.832983,1,0
3,ID4,55.423430,43.813190,1,0
4,ID5,55.381287,43.811452,1,0


Проверка на специально испорченном наборе: координаты вне диапазона,
текст вместо числа, дробная и недопустимая частота, дубликат, пустой
менеджер.

In [3]:
broken = pd.DataFrame({
    "point_id": ["A", "B", "C", "D", "E", "F", "A"],
    "lat": [56.3, 95.0, "abc", 56.1, 56.2, 56.2, 56.3],
    "lon": [44.0, 44.0, 44.0, 44.1, 44.2, 44.3, 44.0],
    "n_visits": [2, 1, 1, 1.5, 5, 1, 1],
    "manager": [0, 0, 1, 1, 1, None, 0],
})
ok, bad = load_and_validate_data(broken, cfg.allowed_visits)
display(ok)
bad[["source_row", "point_id", "reason"]]

Переименованы колонки: {'lat': 'latitude', 'lon': 'longitude', 'n_visits': 'visits_per_month'}


1 строк без manager — назначен 0


Загружено строк: 7, корректных: 2, отклонено: 5


  отклонено 1: latitude вне [-90, 90]


  отклонено 1: latitude не является числом


  отклонено 1: visits_per_month не целое число


  отклонено 1: visits_per_month вне допустимого списка [1, 2, 3, 4]


  отклонено 1: дубликат point_id


,point_id,latitude,longitude,visits_per_month,manager
0,A,56.3,44.0,2,0
1,F,56.2,44.3,1,0


,source_row,point_id,reason
0,3,B,"latitude вне [-90, 90]"
1,4,C,latitude не является числом
2,5,D,visits_per_month не целое число
3,6,E,"visits_per_month вне допустимого списка [1, 2,..."
4,8,A,дубликат point_id


## 2. Потребность и ёмкость

Самое важное наблюдение о данных: визитов больше, чем помещается в схему
«один кластер ≤ 12 точек = один день менеджера».

In [4]:
summary = summarize_points(points)
summary["ёмкость_1_кластер_в_день"] = cfg.working_days * 1 * cfg.max_points_per_cluster
summary["ёмкость_2_кластера_в_день"] = (cfg.working_days
                                       * cfg.max_clusters_per_manager_per_day
                                       * cfg.max_points_per_cluster)
summary["визитов_в_день"] = (summary["visits_required"] / cfg.working_days).round(1)
summary

,manager,points,visits_required,points_1x,points_2x,ёмкость_1_кластер_в_день,ёмкость_2_кластера_в_день,визитов_в_день
0,0,215,405,25,190,264,528,18.4
1,1,206,326,86,120,264,528,14.8
2,2,212,394,30,182,264,528,17.9


Требуется 1125 визитов (15–18 в день на менеджера), а при одном
кластере в день помещается только 3 × 22 × 12 = 792. Поэтому v2 отбрасывала
«лишние» кластеры и теряла визиты. В v3 менеджер может пройти до
`max_clusters_per_manager_per_day = 2` кластеров в день, и загрузка
составляет 62–77 % ёмкости. Если ёмкости не хватит (например, при другом
наборе данных), визиты не исчезнут, а попадут в `unassigned.csv` с
причиной.

## 3. Кластеризация

### Что было с DBSCAN в v2
Метрика `haversine` в scikit-learn ожидает `eps` **в радианах**, а в v2
передавалось `eps_km / 111.32`, то есть градусы.

In [5]:
from sklearn.cluster import DBSCAN

coords = np.radians(points[["latitude", "longitude"]].to_numpy())
eps_km = 25
wrong = DBSCAN(eps=eps_km / 111.32, min_samples=1, metric="haversine").fit(coords)
right = DBSCAN(eps=eps_km / 6371.0088, min_samples=1, metric="haversine").fit(coords)
print(f"eps как в v2: {eps_km / 111.32:.3f} рад ≈ {eps_km / 111.32 * 6371:.0f} км "
      f"-> кластеров: {len(set(wrong.labels_))}")
sizes = pd.Series(right.labels_).value_counts()
print(f"eps в радианах (25 км) -> кластеров: {len(sizes)}, "
      f"самый большой: {sizes.max()} точек")

eps как в v2: 0.225 рад ≈ 1431 км -> кластеров: 1
eps в радианах (25 км) -> кластеров: 5, самый большой: 581 точек


Даже с исправленными единицами DBSCAN связывает плотную агломерацию в
один огромный кластер: размер он не ограничивает. Поэтому в v3 используется
**KMeans с ограничением ёмкости**. Число кластеров вычисляется автоматически
(`ceil(n / (12 × 0.9))`), а шаг назначения решается как задача о назначениях,
где у каждого центра ровно 12 мест. Лимит размера гарантирован построением.
Координаты предварительно проецируются в километры.

In [6]:
m0 = points[points["manager"] == 0]
labels = cluster_points(m0, max_points=cfg.max_points_per_cluster,
                        fill_ratio=cfg.cluster_fill_ratio,
                        random_state=cfg.random_seed)
print("Размеры территорий менеджера 0:",
      sorted(labels["cluster"].value_counts().tolist(), reverse=True))
again = cluster_points(m0, cfg.max_points_per_cluster,
                       fill_ratio=cfg.cluster_fill_ratio,
                       random_state=cfg.random_seed)
print("Повторяемость при фиксированном seed:", labels.equals(again))

dist = HaversineDistance()
diam = (m0.merge(labels, on="point_id").groupby("cluster")
        .apply(lambda g: dist.matrix(g["latitude"], g["longitude"]).max(),
               include_groups=False))
diam.describe().round(1).rename("диаметр территории, км")

Размеры территорий менеджера 0: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 6, 4, 1]
Повторяемость при фиксированном seed: True


count    20.0
mean      6.5
std       8.4
min       0.0
25%       2.1
50%       3.3
75%       6.5
max      31.7
Name: диаметр территории, км, dtype: float64

## 4. План месяца

`run_pipeline` выполняет весь сценарий: валидацию, кластеризацию,
распределение по дням, маршруты, проверки и сохранение.

**Расстояния.** Порядок обхода, км и часы считаются **по дорогам**: для
каждого дня менеджера один запрос к матрице расстояний OSRM (Table API).
Ответы кешируются в `cache/osrm_table_cache.json`, поэтому повторный
запуск идёт без сети и даёт тот же результат. Первый запуск занимает
около 2 минут. Если OSRM недоступен, расстояния считаются по прямой ×1.3,
и это видно в строке «Источник расстояний». Группировка точек в
территории выполняется по координатам: компактность территории от
источника расстояний не зависит.

Правило дней: точка с N визитами посещается в дни `p + round(k·22/N)`.
Для двух визитов это `p` и `p + 11`, ровно полмесяца. Территория
«переезжает» по этим смещениям: в первый визит обходятся все её точки, во
второй — только точки с 2 визитами. Первые дни территорий подбираются
жадно, а затем локальным поиском, чтобы сгладить пиковые дни и ставить
рядом лежащие территории в один день.

In [7]:
out = run_pipeline(DATA_PATH, cfg, output_dir=OUTPUT_DIR, make_map=True)
result = out["result"]
print("Источник расстояний:", result.distance_source)

Переименованы колонки: {'lat': 'latitude', 'lon': 'longitude', 'n_visits': 'visits_per_month'}


Загружено строк: 633, корректных: 633, отклонено: 0


Менеджер 0: требуется 405 визитов, ёмкость 528 (77%)


Менеджер 1: требуется 326 визитов, ёмкость 528 (62%)


Менеджер 2: требуется 394 визитов, ёмкость 528 (75%)


...

Геометрия дорог для карты (OSRM)...


Карта: D:\denis\geolocation\project\outputs\route_map.html


OSRM: запросов 0, из кеша 65, не получено (прямые) 0


Сохранено: D:\denis\geolocation\project\outputs\road_stats.csv (112 строк)


Источник расстояний: osrm (по дорогам): запросов 0, получено пар 0, из кеша 72180, по прямой×1.3 0 пар (0.0%)


### Насколько дороги отличаются от прямой
Для каждого кластера сравним длину обхода по прямой и по дорогам (OSRM).

In [8]:
road = out["distance"]
straight = HaversineDistance()
rows = []
for cid, grp in result.schedule.groupby("cluster_id"):
    grp = grp.sort_values("order_in_route")
    idx = range(len(grp))
    rows.append({
        "cluster_id": cid,
        "по_прямой_км": path_length(straight.matrix(grp["latitude"], grp["longitude"]), idx),
        "по_дорогам_км": path_length(road.matrix(grp["latitude"], grp["longitude"]), idx),
    })
cmp_road = pd.DataFrame(rows)
print(f"Обход всех кластеров: по прямой {cmp_road['по_прямой_км'].sum():.0f} км, "
      f"по дорогам {cmp_road['по_дорогам_км'].sum():.0f} км "
      f"(в {cmp_road['по_дорогам_км'].sum() / cmp_road['по_прямой_км'].sum():.1f} раза больше)")
cmp_road.describe().round(1)

Обход всех кластеров: по прямой 860 км, по дорогам 1483 км (в 1.7 раза больше)


,по_прямой_км,по_дорогам_км
count,112.0,112.0
mean,7.7,13.2
std,6.1,9.5
min,0.0,0.0
25%,4.0,7.5
50%,6.0,10.0
75%,8.9,15.5
max,32.1,51.5


Внутри города и посёлков дорога заметно длиннее прямой: мешают река,
мосты, односторонние улицы и тупики. Поэтому план строится по дорожным
расстояниям, а не по прямой с усреднённым коэффициентом.

### Проверки по критериям приёмки ТЗ
Последняя проверка информативная: оценка длительности дня по средней
скорости.

In [9]:
result.checks

,check,passed,details
0,Для каждой точки учтено ровно visits_per_month...,True,расхождений: 0
1,Все визиты назначены (нет дефицита),True,"назначено 1125 из 1125, дефицит 0"
2,Все точки распределены по кластерам,True,без кластера: 0
3,Размер кластера <= 12,True,"макс. размер 12, кластеров 112"
4,У менеджера <= 2 кластеров в день,True,максимум 2
5,Точка не посещается дважды в один день,True,повторов: 0
6,Интервал между визитами точки >= floor(дней / ...,True,нарушений: 0
7,visit_day в диапазоне 1..22,True,
8,order_in_route — перестановка 1..n в каждом кл...,True,
9,Оценка дня <= 9.0 ч (информативно),False,дней сверх нормы: 6; макс. 11.97 ч


### Итоговая таблица (формат ТЗ + служебные поля)

In [10]:
result.schedule.head(12)

,point_id,visit_day,cluster_id,order_in_route,manager,cluster_seq_in_day,visit_number,visits_per_month,latitude,longitude
0,ID195,1,M0-D01-T04,1,0,1,1,2,56.241944,43.521593
1,ID98,1,M0-D01-T04,2,0,1,1,2,56.243450,43.512914
2,ID191,1,M0-D01-T04,3,0,1,1,2,56.244600,43.511226
3,ID100,1,M0-D01-T04,4,0,1,1,2,56.240349,43.504613
4,ID157,1,M0-D01-T04,5,0,1,1,2,56.239979,43.505891
5,ID99,1,M0-D01-T04,6,0,1,1,2,56.241518,43.496908
6,ID102,1,M0-D01-T04,7,0,1,1,2,56.241188,43.504336
7,ID193,1,M0-D01-T04,8,0,1,1,2,56.244536,43.504485
8,ID192,1,M0-D01-T04,9,0,1,1,2,56.244461,43.498540
9,ID101,1,M0-D01-T04,10,0,1,1,2,56.244550,43.490514


In [11]:
# Пример: точка с двумя визитами
pid = points.loc[points["visits_per_month"] == 2, "point_id"].iloc[0]
result.schedule.loc[result.schedule["point_id"] == pid,
                    ["point_id", "visit_day", "cluster_id", "order_in_route",
                     "visit_number", "visits_per_month"]]

,point_id,visit_day,cluster_id,order_in_route,visit_number,visits_per_month
489,ID49,10,M0-D10-T09,11,1,2
1033,ID49,21,M0-D21-T09,7,2,2


In [12]:
gaps = (result.schedule[result.schedule["visits_per_month"] == 2]
        .groupby("point_id")["visit_day"].agg(lambda d: d.max() - d.min()))
print("Интервал между визитами точек с 2 визитами:", gaps.value_counts().to_dict())
print("Кластеров:", len(result.clusters),
      "| размер: мин", result.clusters["n_points"].min(),
      "макс", result.clusters["n_points"].max())
print("Кластеров у менеджера в день:",
      result.daily["clusters"].value_counts().sort_index().to_dict())
print("Дефицит (unassigned):", len(result.unassigned))

Интервал между визитами точек с 2 визитами: {11: 492}
Кластеров: 112 | размер: мин 1 макс 12
Кластеров у менеджера в день: {1: 18, 2: 47}
Дефицит (unassigned): 0


### Загрузка по дням (оценка часов)

In [13]:
result.daily.pivot(index="visit_day", columns="manager",
                   values="est_hours").round(1).T

visit_day,1,2,3,4,5,6,7,8,9,10,...,13,14,15,16,17,18,19,20,21,22
manager,,,,,,,,,,,,,,,,,,,,,
0,4.8,5.7,4.2,5.3,8.2,7.4,7.1,6.7,6.9,8.3,...,5.7,9.3,9.7,7.6,6.7,6.8,5.7,6.9,6.6,5.9
1,3.6,7.2,6.1,3.7,6.8,3.7,3.9,6.7,12.0,3.6,...,5.0,4.5,2.3,5.0,9.2,3.3,5.5,11.5,2.8,4.2
2,4.7,9.0,10.7,3.8,7.2,7.3,7.0,7.4,7.2,7.0,...,8.7,NaN,3.5,6.7,5.9,6.0,7.1,6.6,6.5,6.7


In [14]:
over = result.daily[result.daily["over_norm"]]
over_clusters = result.clusters.merge(over[["visit_day", "manager"]])
over_clusters[["cluster_id", "n_points", "route_km", "approach_km",
               "return_km", "est_hours"]]

,cluster_id,n_points,route_km,approach_km,return_km,est_hours
0,M2-D03-T02,11,20.98,186.04,189.60,10.68
1,M1-D09-T01,5,9.00,187.56,0.00,5.18
2,M1-D09-T02,10,21.20,7.79,185.28,6.79
3,M0-D14-T07,4,6.38,114.95,0.00,3.43
4,M0-D14-T17,11,29.71,112.75,13.41,5.87
5,M0-D15-T00,6,41.66,60.61,0.00,3.55
6,M0-D15-T05,1,0.00,183.81,113.19,6.19
7,M1-D17-T04,2,2.31,186.60,0.00,4.28
8,M1-D17-T12,4,3.13,177.11,14.74,4.90
9,M1-D20-T03,4,3.65,157.02,0.00,4.21


Дни сверх нормы 9 ч — это выезды к территориям в 100–160 км от центра
Нижнего Новгорода (`approach_km`). Один такой выезд туда и обратно уже
занимает большую часть дня, поэтому алгоритм распределения эти дни не
исправит. Это вопрос к бизнесу: другая точка старта для удалённых
территорий, командировка или реалистичная скорость на трассе
(`avg_speed_kmh`).

## 5. Маршруты: ближайший сосед vs + 2-opt

`build_route` строит обход от точки старта методом «ближайший сосед» и
улучшает его 2-opt. Сравним длину обхода внутри кластеров по дорогам.

In [15]:
rows = []
for cid, grp in result.schedule.groupby("cluster_id"):
    pts = grp[["point_id", "latitude", "longitude"]]
    if len(pts) < 3:
        continue
    m = road.matrix(pts["latitude"], pts["longitude"])
    idx = {p: i for i, p in enumerate(pts["point_id"])}
    nn = build_route(pts, road, start=None, two_opt=False)
    opt = build_route(pts, road, start=None, two_opt=True)
    rows.append({"nn_km": path_length(m, [idx[p] for p in nn]),
                 "two_opt_km": path_length(m, [idx[p] for p in opt])})
cmp = pd.DataFrame(rows)
print(f"Сумма по {len(cmp)} кластерам: NN {cmp.nn_km.sum():.1f} км, "
      f"NN+2-opt {cmp.two_opt_km.sum():.1f} км "
      f"(-{(1 - cmp.two_opt_km.sum() / cmp.nn_km.sum()) * 100:.1f}%)")

Сумма по 107 кластерам: NN 1405.4 км, NN+2-opt 1379.4 км (-1.8%)


## 6. Карта

Полная карта сохранена в `outputs/route_map.html`: каждый день каждого
менеджера — отдельный слой, легенда на карте. **Линии проходят по дорогам**
(OSRM Route API, ответы кешируются в `cache/osrm_cache.json`). Километры
на карте совпадают с планом, в подсказках к линиям указаны км и минуты.
Ниже — один день для примера. Цвет обозначает кластер дня, цифра — `order_in_route` из
итоговой таблицы, линии соединяют точки строго в этом порядке. Пунктир —
подъезд, переезд между кластерами и возврат.

In [16]:
day1 = result.schedule[result.schedule["visit_day"] == 1]
day_map, day_road = visualize_map(
    points[points["point_id"].isin(day1["point_id"])], day1,
    depot=cfg.depot, router=OsrmRouter(OSRM_CACHE_PATH))
display(day_road)
day_map

,manager,visit_day,cluster_id,road_km,road_min,transfer_km,transfer_min
0,0,1,M0-D01-T04,11.1,23.1,79.1,85.9
1,1,1,M1-D01-T19,15.4,31.1,13.6,20.1
2,2,1,M2-D01-T04,14.1,29.2,84.8,92.4


In [17]:
# Сверка: км по плану и по линиям карты (OSRM Route API)
road_stats = pd.read_csv(OUTPUT_DIR / "road_stats.csv")
chk = result.clusters.merge(road_stats, on=["cluster_id", "manager", "visit_day"])
print(f"Обход: план {chk['route_km'].sum():.0f} км, карта {chk['road_km'].sum():.0f} км")
print(f"Всего с подъездами: план "
      f"{(chk['route_km'] + chk['approach_km'] + chk['return_km']).sum():.0f} км, "
      f"карта {(chk['road_km'] + chk['transfer_km']).sum():.0f} км")

Обход: план 1482 км, карта 1482 км
Всего с подъездами: план 6324 км, карта 6323 км


## 7. Сравнение с v2

In [18]:
old_path = PROJECT_DIR.parent / "outputs2" / "schedule_final.csv"
if old_path.exists():
    old = pd.read_csv(old_path)
    req = points.set_index("point_id")["visits_per_month"]
    got_old = old.groupby("point_id").size().reindex(req.index, fill_value=0)
    got_new = result.schedule.groupby("point_id").size().reindex(req.index,
                                                                 fill_value=0)
    comparison = pd.DataFrame({
        "v2": [len(old), int((got_old == 0).sum()),
               int((req - got_old).clip(lower=0).sum())],
        "v3": [len(result.schedule), int((got_new == 0).sum()),
               int((req - got_new).clip(lower=0).sum())],
    }, index=["назначено визитов", "точек без визита", "потеряно визитов"])
    display(comparison)
else:
    print("Результаты v2 не найдены:", old_path)

,v2,v3
назначено визитов,777,1125
точек без визита,85,0
потеряно визитов,348,0


## Выводы

* **Все 1125 визитов назначены**, ни одна точка не потеряна (в v2 было
  потеряно 348 визитов, 85 точек не посещались вовсе).
* Все проверки по критериям приёмки ТЗ пройдены. Кластеры ≤ 12 точек, у
  менеджера ≤ 2 кластеров в день, точки с двумя визитами посещаются
  ровно через 11 рабочих дней, `order_in_route` согласован с картой.
* Результат воспроизводим при фиксированном `random_seed`: это проверяется
  тестами `tests/`.
* Некорректные входные строки не исправляются молча, а попадают в
  `rejected_rows.csv`. Дефицит ёмкости, если он возникнет, попадает в
  `unassigned.csv`.
* Маршруты, км и часы считаются по дорогам (OSRM, с кешем), линии на
  карте проходят по дорогам и совпадают с планом по километрам. Источник
  расстояний явно указывается в логе и `report.json`.

**Открытые вопросы к бизнесу:** точка старта для удалённых территорий,
норматив рабочего дня и реалистичная средняя скорость. Это влияет на 6
дней, которые по оценке превышают 9 ч.